# Churn Risk-Adjusted Customer Value in Retail Banking

## Implementation

This notebook contains the implementation of the churn risk-adjusted Customer Value framework, including data preparation, baseline Customer Value estimation, churn prediction modelling, model evaluation and integration of predicted churn risk with Customer Value.

In [ ]:
# Loading packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [ ]:
# 6a: Data Loading and Initial Inspection

# Loading data
data_path = Path("../data/Bank Customer Churn Prediction.csv")
df = pd.read_csv(data_path)

# Inspecting the data

In [10]:
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
df.shape

(10000, 12)

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  str    
 3   gender            10000 non-null  str    
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), str(2)
memory usage: 937.6 KB


In [13]:
df.columns.tolist()

['customer_id',
 'credit_score',
 'country',
 'gender',
 'age',
 'tenure',
 'balance',
 'products_number',
 'credit_card',
 'active_member',
 'estimated_salary',
 'churn']

In [14]:
df.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df["churn"].value_counts()

churn
0    7963
1    2037
Name: count, dtype: int64

In [17]:
df["churn"].value_counts(normalize=True)

churn
0    0.7963
1    0.2037
Name: proportion, dtype: float64

In [18]:
df.describe()

,customer_id,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [20]:
# Unique values in categorical/binary variables
for col in ["country", "gender", "credit_card", "active_member", "products_number", "churn"]:
    print(f"\n{col}")
    print(df[col].value_counts())


country
country
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

gender
gender
Male      5457
Female    4543
Name: count, dtype: int64

credit_card
credit_card
1    7055
0    2945
Name: count, dtype: int64

active_member
active_member
1    5151
0    4849
Name: count, dtype: int64

products_number
products_number
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

churn
churn
0    7963
1    2037
Name: count, dtype: int64


In [22]:
df["products_number"].value_counts().sort_index()

products_number
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

In [21]:
zero_balance = (df["balance"] == 0).sum()

print("Customers with zero balance:", zero_balance)
print("Percentage with zero balance:", zero_balance / len(df) * 100)

Customers with zero balance: 3617
Percentage with zero balance: 36.17


In [23]:
pd.crosstab(
    df["products_number"],
    df["balance"] == 0,
    margins=True
)

balance,False,True,All
products_number,,,
1,4179,905,5084
2,1990,2600,4590
3,168,98,266
4,46,14,60
All,6383,3617,10000


### Key  notes from the initial inspection findings

The dataset contains 10K customers and 12 variables

-No missing values.

-No duplicate customer records.

-The churn variable is imbalanced, with 7,963 (79.63%) non-churners and 2,037 (20.37%) churners.

-3,617 customers (36.17%) have a zero account balance.

-Product ownership ranges from 1-4. Most customers hold either one (5,084) or two (4,590) products.

-of the zero balance customers, 2,600 hold two products, while 112 customers hold three or four products.

The design of the baseline Customer Value measure will need to be modified due to high number of zero balance customers. A purely multiplicative combination of balance and product ownership would assign a CV score of zero to all zero-balance customers regardless of their product holdings. Therefore, the method used to combine balance and product ownership will be investigated further before the final CV calculation is implemented.